### Uvoz biblioteka

In [19]:
import os
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

# Podesavanje prikaza
%matplotlib inline
print("Biblioteke su uspjesno ucitane!")

Biblioteke su uspjesno ucitane!


### Definisanje klasa 

In [20]:
CLASSES = {
    "Hydrolase": "hydrolase.tsv",
    "Transport protein": "transport_protein.tsv",
    "Transcription factor": "transcription_factor.tsv",
    "Receptor": "receptor.tsv",
    "Structural protein": "structural_protein.tsv"
}

### Učitavanje i spajanje TSV fajlova

In [21]:
all_dfs = []
for label, filename in CLASSES.items():
    df = pd.read_csv(f"../data/{filename}", sep="\t")
    df["label"] = label
    print(f"{label}: {len(df)} proteina")
    all_dfs.append(df)

dataset = pd.concat(all_dfs, ignore_index=True)
print(f"\nUkupno učitano proteina: {len(dataset)}")

Hydrolase: 2407 proteina
Transport protein: 1242 proteina
Transcription factor: 1418 proteina
Receptor: 1605 proteina
Structural protein: 774 proteina

Ukupno učitano proteina: 7446


### Upoznavanje sa skupom podataka

In [22]:
# Dimenzije i kolone
print(f"Dimenzije dataseta: {dataset.shape}")
print(f"Broj klasa: {dataset['label'].nunique()}")
print(f"Kolone: {list(dataset.columns)}\n")

Dimenzije dataseta: (7446, 5)
Broj klasa: 5
Kolone: ['Entry', 'Sequence', 'Protein names', 'Keywords', 'label']



In [23]:
# Prikaz prvih redova
print("Prvih 5 redova:")
dataset.head()

Prvih 5 redova:


,Entry,Sequence,Protein names,Keywords,label
0,A0A1B0GTW7,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,Ciliated left-right organizer metallopeptidase...,Alternative splicing;Disease variant;Glycoprot...,Hydrolase
1,A1A4Y4,MEAMNVEKASADGNLPEVISNIKETLKIVSRTPVNITMAGDSGNGM...,Immunity-related GTPase family M protein (EC 3...,Alternative splicing;Autophagy;Cell membrane;C...,Hydrolase
2,A1KZ92,MEPRLFCWTTLFLLAGWCLPGLPCPSRCLCFKSTVRCMHLMLDHIP...,Probable oxidoreductase PXDNL (EC 1.-.-.-) (Ca...,Alternative splicing;Calcium;Cell membrane;Cyt...,Hydrolase
3,A1Z1Q3,MYPSNKKKKVWREEKERLLKMTLEERRKEYLRDYIPLNSILSWKEE...,ADP-ribose glycohydrolase MACROD2 (MACRO domai...,3D-structure;Alternative splicing;DNA damage;H...,Hydrolase
4,A2A288,MEHPSKMEFFQKLGYDREDVLRVLGKLGEGALVNDVLQELIRTGSR...,Probable ribonuclease ZC3H12D (EC 3.1.-.-) (MC...,Alternative splicing;Chromosomal rearrangement...,Hydrolase


In [24]:
# Provjera nedostajucih vrijednosti
print("Broj nedostajucih vrijednosti po kolonama:")
print(dataset.isnull().sum())

Broj nedostajucih vrijednosti po kolonama:
Entry            0
Sequence         0
Protein names    0
Keywords         0
label            0
dtype: int64


In [25]:
# Identifikovanje dupliranih sekvenci i multifunkcionalnih proteina
dup_count = dataset['Sequence'].duplicated(keep=False).sum()
unique_seq_count = dataset['Sequence'].nunique()

print(f"Ukupno unikatnih sekvenci: {unique_seq_count}")
print(f"Broj redova sa dupliranim sekvencama: {dup_count}")

Ukupno unikatnih sekvenci: 7044
Broj redova sa dupliranim sekvencama: 788


### Pretprocesiranje i čišćenje sekvenici

Uklanjamo prazne redove, sekvence kraće od 10 aminokiselina i sekvence koje sadrže nestandardne aminokiseline (B, J, O, U, X, Z).

In [26]:
# Redovi sa praznom sekvencom
print(f"Broj redova sa praznom sekvencom: {dataset['Sequence'].isna().sum()}")

Broj redova sa praznom sekvencom: 0


In [27]:
def valid_sequence(seq):
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    seq = str(seq).upper().strip()

    unvalid = [aa for aa in seq if aa not in valid_aa]
    if unvalid:
        print(f"Nevalidni karakteri: {set(unvalid)}")
        return None
    
    if len(seq) < 10:
        print(f"Prekratka sekvenca: {len(seq)}")
        return None
    
    return seq

# Validacija
dataset['Sequence'] = dataset['Sequence'].apply(valid_sequence)

total_before = len(dataset)
dataset = dataset.dropna(subset=['Sequence'])
total_after = len(dataset)

print(f"Prije odbacivanja: {total_before} proteina")
print(f"Nakon odbacivanja: {total_after} proteina")
print(f"Uklonjeno nevalidnih sekvenci: {total_before - total_after}")

Nevalidni karakteri: {'U'}
Prije odbacivanja: 7446 proteina
Nakon odbacivanja: 7445 proteina
Uklonjeno nevalidnih sekvenci: 1


### Formiranje Single-Class (SC) i Multi-Class (MC) skupova

Grupišemo klase po unikatnim sekvencama i dijelimo podatke na dva skupa:
- **SC (Single-Class):** Sekvence sa tačno 1 funkcionalnom klasom
- **MC (Multi-Class):** Sekvence sa 2 ili više funkcionalnih klasa

In [31]:
# Grupisanje klasa po sekvenci
grouped = dataset.groupby('Sequence')['label'].apply(lambda labels: sorted(list(set(labels)))).reset_index()
grouped['num_classes'] = grouped['label'].apply(len)

# 1. Single-Class (SC) skup
df_sc = grouped[grouped['num_classes'] == 1].copy()
df_sc['label'] = df_sc['label'].apply(lambda x: x[0])

# 2. Multi-Class (MC) skup
df_mc = grouped[grouped['num_classes'] > 1].copy()

print(f"1. Single-Class (SC) skup: {len(df_sc)} proteina")
print(f"2. Multi-Class (MC) skup:  {len(df_mc)} proteina")

1. Single-Class (SC) skup: 6663 proteina
2. Multi-Class (MC) skup:  380 proteina


### Čuvanje obrađenih podataka

In [ ]:
processed_dir = os.path.join("..", "data", "processed")
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir, exist_ok=True)

# Čuvanje SC skupa
sc_path = os.path.join(processed_dir, "df_sc_cleaned.csv")
df_sc[['Sequence', 'label']].to_csv(sc_path, index=False)

# Čuvanje MC skupa (spajamo listu klasa sa '|')
df_mc_to_save = df_mc.copy()
df_mc_to_save['label'] = df_mc_to_save['label'].apply(lambda x: '|'.join(x))
mc_path = os.path.join(processed_dir, "df_mc_cleaned.csv")
df_mc_to_save[['Sequence', 'label', 'num_classes']].to_csv(mc_path, index=False)

print("Podaci su uspešno sačuvani u 'data/processed/':")
print(f" - {sc_path}")
print(f" - {mc_path}")

Podaci su uspešno sačuvani u 'data/processed/':
 - ../data\processed\df_sc_cleaned.csv
 - ../data\processed\df_mc_cleaned.csv
